# 05 RetrievalRouter on the ColVec depth ladder (Kaggle T4)

Runs the RetrievalRouter training code (emrekuruu/retrieval-router, pinned commit) on our ladder labels, 5-fold CV on our fixed folds (seed 42).

- Arms: `LIGHT`, `VISUAL20`, `VISUAL50`, `VISUAL100` (ColVec re-sort of light's top 20 / 50 / 100).
- Their `_ndcg` column carries our R@10 (0-1), their `_latency` column carries ColVec pages (0 / 20 / 50 / 100), per-row normalised by their own `calculate_rewards`.
- Jobs: their `retrievalrouter` objective at each lambda (decision = argmax, as their `evaluate_model`), and their Arabzadeh `baseline` objective at each rank threshold T.
- The baseline's decision is their probability cutoff (`select_at_tau`), not argmax, so its out-of-fold probabilities go to `probs/`; `python -m ondemand.routing_ladder arabzadeh` applies their budget matching locally.
- Only `arms.py` (arm list) and the bf16 check in `train/config.py` (T4 has no bf16) are patched; the diff is printed.

Input: dataset with `labels_ladder_k100.jsonl`. Settings: Accelerator GPU T4, Internet ON.
Output: `/kaggle/working/routing_out_05.zip`.

## 1. Settings

In [ ]:
INPUT_NAME = "labels_ladder_k100.jsonl"
OUT = "/kaggle/working/routing_out"
ZIP = "routing_out_05.zip"
REPO = "https://github.com/emrekuruu/retrieval-router"
COMMIT = "6fca6dd498161d0a49820b1b0b45060b1e976306"
CHECKOUT = "/kaggle/working/rr_ladder"

ARM_PAGES = {"LIGHT": 0.0, "VISUAL20": 20.0, "VISUAL50": 50.0, "VISUAL100": 100.0}
LAMBDAS = [0.0, 0.1, 0.3, 0.5, 0.7]     # their evaluation grid (evaluation/run_adaptive_significance.py); config default 0.1
RANK_THRESHOLDS = [1, 2]                # baseline T; their faithful default 1, 2 = their sensitivity run
VAL_RATE = 0.1                          # their SAMPLE_RATE, drawn from the training folds per group
SEED = 42                               # their RANDOM_SEED
HF_AUTHOR = "emrekuruu"
ORIG = "/kaggle/working/rr_orig"
RUN_ZERO_SHOT = True
RUN_RETRAIN = False


## 2. Install, clone, patch
Kaggle's preinstalled torch is kept (do not reinstall torch). Kaggle's torchao is removed: neither repo uses it and peft rejects the old version. Versions follow their `requirements/training_requirements.txt`; transformers is held at 4.56-4.57: their `train.py` loads the model with `dtype=` (4.56+), and their code was written against 4.x.

In [ ]:
!pip uninstall -y -q torchao
TRANSFORMERS = '"transformers>=4.56,<4.58"' if RUN_RETRAIN else '"transformers>=5,<5.15"'
!pip install -q {TRANSFORMERS} "peft>=0.14,<0.20" "accelerate>=1.2" "datasets>=3.0" "openpyxl>=3.1" tensorboard scikit-learn


In [ ]:
import glob
import json
import os
import subprocess
import sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("no HF_TOKEN secret; gated models will fail to download")

found = sorted(glob.glob(f"/kaggle/input/**/{INPUT_NAME}", recursive=True))
if not found:
    raise SystemExit(f"attach a dataset containing {INPUT_NAME} (from data/work/ondemand_v2/<fp>/routing/)")
INPUT = found[0]
rows = [json.loads(line) for line in open(INPUT)]
print(INPUT, len(rows), "queries", sorted({r["fold"] for r in rows}), "folds")
if len(rows) not in (214, 220) or "visual100_recall@10" not in rows[0]:
    raise SystemExit("wrong input file: expected the ladder labels (214 queries, or 220 before the 6 flawed were dropped)")
os.makedirs(f"{OUT}/decisions", exist_ok=True)
os.makedirs(f"{OUT}/logs", exist_ok=True)

import torch
if not torch.cuda.is_available():
    raise SystemExit("no GPU: set Accelerator to GPU T4 in the notebook settings")
print(sys.version.split()[0], "torch", torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
def run(cmd, log, cwd, env=None):
    with open(log, "w") as sink:
        code = subprocess.run(cmd, cwd=cwd, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONPATH": cwd, **(env or {})},
                              stdout=sink, stderr=subprocess.STDOUT).returncode
    if code:
        print(open(log).read()[-3000:])
        raise SystemExit(f"failed ({code}): {' '.join(cmd)}; full log in {log}")

import re

if not os.path.exists(CHECKOUT):
    subprocess.run(["git", "clone", "-q", REPO, CHECKOUT], check=True)
subprocess.run(["git", "-C", CHECKOUT, "checkout", "-q", "--", "."], check=True)
subprocess.run(["git", "-C", CHECKOUT, "checkout", "-q", COMMIT], check=True)

names = list(ARM_PAGES)
arms = open(f"{CHECKOUT}/arms.py").read()
arms = re.sub(r"ARM_NAMES = \[.*?\]", "ARM_NAMES = [\n" + "".join(f'    "{a}",\n' for a in names) + "]", arms, count=1, flags=re.S)
arms = re.sub(r"NON_ARM_PIPELINES = \[.*?\]", "NON_ARM_PIPELINES = []", arms, count=1, flags=re.S)
arms = arms.replace("ARM_SHORT = ", 'SHORT_NAMES.update({"LIGHT": "LI", "VISUAL20": "V20", "VISUAL50": "V50", "VISUAL100": "V100"})\n'
                    'DISPLAY_NAMES.update({"LIGHT": "Light", "VISUAL20": "Visual@20", "VISUAL50": "Visual@50", "VISUAL100": "Visual@100"})\n'
                    'COLORS.update({"LIGHT": "#F4A460", "VISUAL20": "#FF6B35", "VISUAL50": "#C41E3A", "VISUAL100": "#8B0000"})\n\nARM_SHORT = ', 1)
open(f"{CHECKOUT}/arms.py", "w").write(arms)

config = open(f"{CHECKOUT}/train/config.py").read()
old = "USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()"
if old not in config:
    raise SystemExit("bf16 patch target not found")
config = config.replace(old, "USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8")
open(f"{CHECKOUT}/train/config.py", "w").write(config)
print(subprocess.run(["git", "-C", CHECKOUT, "diff"], capture_output=True, text=True).stdout)
if not os.path.exists(ORIG):
    subprocess.run(["git", "clone", "-q", REPO, ORIG], check=True)
subprocess.run(["git", "-C", ORIG, "checkout", "-q", "--", "."], check=True)
subprocess.run(["git", "-C", ORIG, "checkout", "-q", COMMIT], check=True)
print("rr_orig diff:", repr(subprocess.run(["git", "-C", ORIG, "diff"], capture_output=True, text=True).stdout))


## 3. Preflight: their modules import, arms and precision are as expected

In [ ]:
check = """
import torch, transformers, peft, accelerate, datasets
from arms import ARM_NAMES, NDCG_COLS, LATENCY_COLS
from train import train
from train.config import USE_BF16, DTYPE, MODEL_NAME
from train.inference import run_inference
from train.objectives import get_objective
from peft.import_utils import is_torchao_available
is_torchao_available()
print("transformers", transformers.__version__, "peft", peft.__version__, "accelerate", accelerate.__version__, "datasets", datasets.__version__)
print(ARM_NAMES, "bf16", USE_BF16, DTYPE, MODEL_NAME)
assert ARM_NAMES == %r and not USE_BF16
""" % (list(ARM_PAGES),)
run(["python", "-c", check], f"{OUT}/logs/preflight.log", CHECKOUT)
print(open(f"{OUT}/logs/preflight.log").read()[-2000:])

## 4. Fold driver
Thin wrapper around their `train.main`, `run_inference` and objectives; only the input table is ours.

In [ ]:
%%writefile /kaggle/working/rr_fold.py
import json
import os
import shutil
import sys

import pandas as pd

from arms import ARM_NAMES
from train import train as trainer
from train.config import OBJECTIVE
from train.inference import run_inference
from train.objectives import get_objective

source, fold, workdir, out, pages, val_rate, seed = sys.argv[1:8]
fold, pages, val_rate, seed = int(fold), json.loads(pages), float(val_rate), int(seed)
rows = [json.loads(line) for line in open(source)]
table = []
for r in rows:
    row = {"query_id": r["query_id"], "query": r["query"], "DATASET": r["group"], "fold": r["fold"]}
    for arm in ARM_NAMES:
        row[f"{arm}_ndcg"] = r[f"{arm.lower()}_recall@10"] / 100
        row[f"{arm}_mrr"] = r[f"{arm.lower()}_mrr"]
        row[f"{arm}_latency"] = pages[arm]
    table.append(row)
df = pd.DataFrame(table)
test = df[df["fold"] == fold].reset_index(drop=True)
rest = df[df["fold"] != fold]
val = pd.concat([g.sample(n=max(1, int(len(g) * val_rate)), random_state=seed) for _, g in rest.groupby("DATASET")])
train = rest.drop(val.index).reset_index(drop=True)
val = val.reset_index(drop=True)

objective = get_objective(OBJECTIVE)
for part in (train, val, test):
    objective.add_targets(part)
os.makedirs(workdir, exist_ok=True)
paths = tuple(os.path.join(workdir, f"{name}.xlsx") for name in ("train", "val", "test"))
for part, path in zip((train, val, test), paths):
    part.to_excel(path, index=False)

model, tokenizer = trainer.main(objective, paths=paths, output_dir=workdir)
probs = run_inference(model, tokenizer, test)
with open(out, "w") as sink:
    for qid, p in zip(test["query_id"], probs):
        arm = ARM_NAMES[int(p.argmax())]
        sink.write(json.dumps({"query_id": qid, "branch": arm.lower(), "fold": fold,
                               "probs": dict(zip(ARM_NAMES, map(float, p)))}) + "\n")
shutil.rmtree(workdir)

## A. Released checkpoints, no training
Every model published by `emrekuruu`, run with their `load_model` + `run_inference` on the unpatched clone; argmax over their `ARM_NAMES` (older 4-class models use the first 4). MULTIMODAL_* -> visual, TEXT_* / BM25 -> light.

In [ ]:
%%writefile /kaggle/working/zero_shot.py
import json
import sys

import pandas as pd

from arms import ARM_NAMES
from train.inference import load_model, run_inference

model_id, source, out = sys.argv[1:4]
rows = [json.loads(line) for line in open(source)]
model, tokenizer = load_model(model_id)
probs = run_inference(model, tokenizer, pd.DataFrame({"query": [r["query"] for r in rows]}))
names = ARM_NAMES[:probs.shape[1]]
with open(out, "w") as sink:
    for r, p in zip(rows, probs):
        arm = names[int(p.argmax())]
        sink.write(json.dumps({"query_id": r["query_id"], "branch": "visual" if arm.startswith("MULTIMODAL") else "light",
                               "arm": arm, "probs": dict(zip(names, map(float, p)))}) + "\n")

In [ ]:
from huggingface_hub import HfApi

models = [m.id for m in HfApi().list_models(author=HF_AUTHOR) if m.id.split("/")[-1].startswith("RetrievalRouter")] if RUN_ZERO_SHOT else []
print(models)
for model_id in models:
    name = model_id.split("/")[-1]
    out = f"{OUT}/decisions/rr-zeroshot_{name}.jsonl"
    if os.path.exists(out):
        continue
    try:
        run(["python", "/kaggle/working/zero_shot.py", model_id, INPUT, out], f"{OUT}/logs/zeroshot_{name}.log", ORIG)
        print("done", model_id)
    except SystemExit as error:
        print("skipped", model_id, error)

## 5. 5-fold CV
Resumable: a finished (job, fold) is skipped. Each fold's checkpoint is deleted after its predictions are written (Kaggle disk).

In [ ]:
if RUN_RETRAIN:
    jobs = [(f"rr-ladder-l{int(lam * 100):02d}", {"OBJECTIVE": "retrievalrouter", "LAMBDA": str(lam)}, "decisions") for lam in LAMBDAS]
    jobs += [(f"rr-ladder-baseline-t{t}", {"OBJECTIVE": "baseline", "RANK_THRESHOLD": str(t), "LAMBDA": "0.1"}, "probs") for t in RANK_THRESHOLDS]
    for tag, env, kind in jobs:
        parts = []
        for fold in sorted({r["fold"] for r in rows}):
            part = f"{OUT}/folds/{tag}_fold{fold}.jsonl"
            os.makedirs(os.path.dirname(part), exist_ok=True)
            if not os.path.exists(part):
                run(["python", "/kaggle/working/rr_fold.py", INPUT, str(fold), f"/kaggle/working/tmp_{tag}_{fold}", part,
                     json.dumps(ARM_PAGES), str(VAL_RATE), str(SEED)],
                    f"{OUT}/logs/{tag}_fold{fold}.log", CHECKOUT, env)
                print("done", tag, "fold", fold)
            parts.append(part)
        os.makedirs(f"{OUT}/{kind}", exist_ok=True)
        with open(f"{OUT}/{kind}/{tag}.jsonl", "w") as sink:
            for part in parts:
                sink.write(open(part).read())

## 6. Summary and download
The summary covers `decisions/` (RetrievalRouter, argmax). The baseline files in `probs/` are scored after the cutoff step.

In [ ]:
by_qid = {r["query_id"]: r for r in rows}
alias = {"visual": "visual100"}
pages = {"light": 0, "visual20": 20, "visual50": 50, "visual100": 100}
for path in sorted(glob.glob(f"{OUT}/decisions/*.jsonl")):
    decisions = [json.loads(line) for line in open(path)]
    arms = [alias.get(d["branch"], d["branch"]) for d in decisions]
    recall = sum(by_qid[d["query_id"]][f"{a}_recall@10"] for d, a in zip(decisions, arms)) / len(decisions)
    counts = {a: arms.count(a) for a in pages}
    print(f"{os.path.basename(path):45s} n={len(decisions)} R@10={recall:.2f} pages={sum(pages[a] for a in arms) / len(arms):.1f} {counts}")
subprocess.run(["zip", "-qr", f"/kaggle/working/{ZIP}", "routing_out"], cwd="/kaggle/working", check=True)
print(f"download /kaggle/working/{ZIP} (Output panel on the right)")